# Real 8B QLoRA experiment ? no pre-filled results
Select a GPU runtime in Colab or Kaggle. Upload sql-8b-source.zip alongside this notebook (Kaggle: add it as a private input dataset and enable Internet).
This run downloads Qwen3-8B and Spider, trains a real NF4 adapter, and evaluates independent base/adapter predictions on the same held-out databases.
Free GPU availability and session length vary. This notebook has NOT been run on a GPU yet. A 100-step run is an initial experiment, not proof of a target accuracy.
Public Spider development data is used for final evaluation; this is single-database execution matching, NOT official Spider test-suite accuracy.


In [ ]:
from pathlib import Path
import os,sys,subprocess,json,zipfile,shutil
WORK=Path("/kaggle/working" if Path("/kaggle/working").exists() else "/content")
os.chdir(WORK)
candidates=list(WORK.glob("sql-8b-source.zip"))+list(Path("/kaggle/input").rglob("sql-8b-source.zip")) if Path("/kaggle/input").exists() else list(WORK.glob("sql-8b-source.zip"))
if not candidates:
    from google.colab import files
    files.upload()
    candidates=list(WORK.glob("sql-8b-source.zip"))
if len(candidates)!=1: raise RuntimeError("Provide exactly one sql-8b-source.zip")
with zipfile.ZipFile(candidates[0]) as z:
    for name in z.namelist():
        if not (WORK/name).resolve().is_relative_to(WORK.resolve()): raise ValueError("Unsafe zip path")
    z.extractall(WORK)
ROOT=WORK/"sql-8b-source";os.chdir(ROOT)
def run(*args):
    p=subprocess.run([sys.executable,*args],text=True)
    if p.returncode: raise RuntimeError("Command failed: "+str(args))
run("-m","pip","install","-r","requirements-ml.txt","bitsandbytes>=0.45,<1")


In [ ]:
import torch,platform,importlib.metadata as metadata
if not torch.cuda.is_available(): raise RuntimeError("Select a GPU runtime before continuing")
info={"gpu":torch.cuda.get_device_name(0),"vram_gib":torch.cuda.get_device_properties(0).total_memory/2**30,
      "python":platform.python_version(),"packages":{k:metadata.version(k) for k in ["torch","transformers","peft","datasets","bitsandbytes"]}}
Path("artifacts").mkdir(exist_ok=True)
Path("artifacts/environment.json").write_text(json.dumps(info,indent=2))
print(info)
if info["vram_gib"]<14: raise RuntimeError("This configuration needs approximately 16 GB or more GPU memory")
from huggingface_hub import HfApi,snapshot_download
MODEL_ID="Qwen/Qwen3-8B"
MODEL_REVISION=HfApi().model_info(MODEL_ID).sha
MODEL=snapshot_download(MODEL_ID,revision=MODEL_REVISION,ignore_patterns=["*.gguf","*.msgpack","*.h5"])
Path("artifacts/model_revision.json").write_text(json.dumps({"id":MODEL_ID,"revision":MODEL_REVISION},indent=2))


## Freeze the splits before training
Training/validation/test database IDs are disjoint. The manifest records dataset revisions, hashes and skipped records. Filtering defines a restricted eligible subset; report its counts alongside scores. Do not tune on the final 100-question test file.


In [ ]:
run("-m","experiments.spider","prepare","--model",MODEL,"--output","artifacts/spider",
    "--train-limit","1000","--val-limit","64","--test-limit","100","--max-length","2048")
print(Path("artifacts/spider/manifest.json").read_text())


## Save the baseline first
Each prediction command runs in a separate process so GPU memory is released before the next phase. Keep the prompts and decoding settings identical.


In [ ]:
run("-m","experiments.spider","predict","--model",MODEL,"--test-file","artifacts/spider/test.jsonl","--output","artifacts/base-predictions.jsonl")
run("-m","experiments.spider","score","--test-file","artifacts/spider/test.jsonl","--predictions","artifacts/base-predictions.jsonl","--output","artifacts/base-score.json")


In [ ]:
# Initial bounded experiment. Extend only after inspecting validation loss, not final test results.
MAX_STEPS=100
run("-m","training.train","--model_id",MODEL,"--train_file","artifacts/spider/train.jsonl",
    "--val_file","artifacts/spider/validation.jsonl","--output_dir","artifacts/adapter",
    "--batch_size","1","--gradient_accumulation_steps","8","--max_seq_length","2048",
    "--max_steps",str(MAX_STEPS),"--epochs","1","--report_to","none")


In [ ]:
run("-m","experiments.spider","predict","--model",MODEL,"--adapter","artifacts/adapter",
    "--test-file","artifacts/spider/test.jsonl","--output","artifacts/adapter-predictions.jsonl")
run("-m","experiments.spider","score","--test-file","artifacts/spider/test.jsonl",
    "--predictions","artifacts/adapter-predictions.jsonl","--output","artifacts/adapter-score.json")
for label in ["base","adapter"]:
    r=json.loads(Path(f"artifacts/{label}-score.json").read_text())
    print(label,{k:r[k] for k in ["num_samples","accuracy_pct","wilson_95_pct","nonempty_accuracy_pct","reference_errors"]})


## Actual model-proposed code repair
The source archive also includes the editor implementation. These three disclosed synthetic probes exercise actual model proposals and bounded test feedback; they do not establish a representative success rate.
Run only in this disposable runtime: generated Python code executes with its user privileges. No personal credentials or mounted private Drive are required.


In [ ]:
run("-m","pip","install","tree-sitter>=0.25,<0.27","tree-sitter-python>=0.25,<0.26","tiktoken>=0.9,<1")
editor=ROOT/"repair-editor"
p=subprocess.run([sys.executable,"-m","benchmarks.model_repair","--model",MODEL,"--allow-local-execution",
                  "--output",str(ROOT/"artifacts/repair-model-probes.json")],cwd=editor)
if p.returncode: raise RuntimeError("Repair benchmark failed; inspect stdout and saved evidence")


## Preserve the actual evidence
Download before the runtime expires. The evidence archive includes model revision, package versions, split manifest, predictions, errors, adapter and training state. It excludes base weights and cached dataset files.
Absolute database paths in split files belong to this runtime; to re-score elsewhere, restore the pinned database mirror and update those paths.


In [ ]:
# Include the small SQLite fixtures so evidence remains independently executable.
for split in ["train","validation","test"]:
    f=Path(f"artifacts/spider/{split}.jsonl")
    records=[json.loads(line) for line in f.read_text().splitlines()]
    for rec in records:
        dest=Path("artifacts/spider/databases")/(rec["db_id"]+".sqlite")
        dest.parent.mkdir(exist_ok=True)
        if not dest.exists(): shutil.copy2(rec["database"],dest)
        rec["database"]=str(dest)
    f.write_text("\n".join(json.dumps(r) for r in records)+"\n")
manifest=Path("artifacts/spider/manifest.json")
data=json.loads(manifest.read_text())
data["original_files_sha256"]=data["files_sha256"]
import hashlib
data["files_sha256"]={s:hashlib.sha256(Path(f"artifacts/spider/{s}.jsonl").read_bytes()).hexdigest() for s in ["train","validation","test"]}
data["database_files_sha256"]={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in Path("artifacts/spider/databases").glob("*.sqlite")}
manifest.write_text(json.dumps(data,indent=2))
archive=shutil.make_archive(str(WORK/"sql-8b-results"),"zip",ROOT,"artifacts")
print(archive)
if str(WORK)=="/content":
    from google.colab import files
    files.download(archive)
